# Task 3: Value of Robustification (Section 3.2)

This notebook implements **Task 3** from the paper: comparing nominal and robust
first-stage designs under three scenarios to quantify the benefit of explicitly
optimising for disruptions.

**Background:**
- The **nominal** solution solves eq. (6), ignoring disruptions entirely.
- The **robust** solution solves the two-stage minimax problem eq. (12) via
  Algorithm 1 (Column-and-Constraint Generation, C&CG).
- Given fixed first-stage decisions $x$ and a realised disruption $\varepsilon$,
  the **second-stage SOCP** (Corollary 1, eq. 15) optimises adaptive pricing
  and allocation to maximise operational profit.

**Metrics defined in this notebook:**

| Metric | Definition |
|--------|------------|
| **VOR** (Value of Robustification) | $\pi^{\rm rob}(\varepsilon^*_{\rm rob}) - \pi^{\rm nom}(\varepsilon^*_{\rm nom})$ under each solution's own worst-case $\varepsilon^*$ |
| **Pre-disruption cost of robustness** | $\pi^{\rm nom}(\varepsilon_0) - \pi^{\rm rob}(\varepsilon_0)$, the "peace-time" profit sacrifice |

where $\varepsilon_0$ denotes the no-disruption scenario ($\varepsilon_j = 1\ \forall j$)
and $\varepsilon^*$ denotes the adversarial worst-case disruption found by the
separation oracle (Corollary 2, eq. 17).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rcflp import (
    instancemaker,
    solve_nominal,
    solve_CCG,
    evaluate_second_stage,
    worst_case_disruption,
    no_disruption_scenario,
    sample_disruptions,
    compute_cost_breakdown,
)

## 1. Configuration

In [ ]:
# Main instance parameters
In, Jn, Rn = 10, 8, 2
V_SCALE, W  = 0.75, 10.0
GAMMA, Hn   = 2, 2
N_SAMPLES   = 50      # Monte Carlo samples for average-case
SEED        = 42
TOL         = 0.01
TIME_LIMIT  = 600
DATA_PATH   = '../dataset.xlsx'

# Sensitivity: vary Gamma
GAMMA_RANGE = [0, 1, 2, 3, 4]

## 2. Solve nominal and robust problems

We first solve the **nominal problem** (eq. 6), which maximises expected profit
ignoring any disruptions:

$$\max_{x \in \mathcal{X}}\ Q(x,\,\varepsilon_0) \tag{6}$$

We then solve the **robust problem** (eq. 12) using Algorithm 1
(Column-and-Constraint Generation):

$$\max_{x \in \mathcal{X}}\ \min_{\varepsilon \in \Xi(\Gamma)}\ Q(x,\,\varepsilon) \tag{12}$$

The C&CG algorithm iterates between a master problem and a separation oracle
until the optimality gap falls below `TOL`. The nominal solution $x^{\rm nom}$
is used as a warm-start for the robust solver.

In [ ]:
inst  = instancemaker(In, Jn, Rn, V_SCALE, W, data_path=DATA_PATH)
nom   = solve_nominal(inst)
x_nom = nom['x_jr']
print(f"Nominal profit: {nom['profit']:,.1f}  ({nom['runtime']:.1f}s)")

ccg   = solve_CCG(inst, GAMMA, Hn, x_init=x_nom, tol=TOL,
                  time_limit=TIME_LIMIT, L_init=-abs(nom['profit'])*2, verbose=True)
x_rob = ccg['x_jr']
print(f"Robust profit (LB): {ccg['profit_LB']:,.1f}  converged={ccg['converged']}  ({ccg['runtime']:.1f}s)")

## 3. First-stage solution comparison

Before comparing profits, we inspect which facilities are opened by each
solution and the resulting fixed-cost investment.  A robust design may open
more or larger facilities (higher capacity tier $r$) to hedge against
disruptions, incurring higher fixed costs in exchange for resilience.

In [ ]:
R = inst['R']
J = inst['J']
nom_open = [(j,r) for (j,r),v in x_nom.items() if v > 0.5]
rob_open = [(j,r) for (j,r),v in x_rob.items() if v > 0.5]
fixed_nom = sum(inst['fixed_cost'][j,r]*x_nom[j,r] for j in J for r in R)
fixed_rob = sum(inst['fixed_cost'][j,r]*x_rob[j,r] for j in J for r in R)
print(f"Nominal opens: {sorted(nom_open)}  fixed={fixed_nom:,.0f}")
print(f"Robust  opens: {sorted(rob_open)}  fixed={fixed_rob:,.0f}")
print(f"Solutions differ: {sorted(nom_open) != sorted(rob_open)}")

## 4. Pre-disruption comparison

We evaluate both solutions under the **no-disruption scenario**
$\varepsilon_0 = (\varepsilon_{j0}=1,\ \varepsilon_{jh}=0\ \forall h>0)$,
meaning all facilities operate at full capacity ($\varepsilon_j = 1\ \forall j$).

For each solution the second-stage SOCP (Corollary 1, eq. 15) is solved to
find the optimal adaptive prices and allocations.  Prices are recovered via
eq. (4): $p_i = v_i (1 - \sum_j y_{ij})$.

The **pre-disruption cost of robustness** measures how much profit the robust
solution sacrifices in normal operating conditions:
$$\text{Pre-disruption cost} = \pi^{\rm nom}(\varepsilon_0) - \pi^{\rm rob}(\varepsilon_0)$$

In [ ]:
eps0 = no_disruption_scenario(inst, Hn)
eval_nom_0 = evaluate_second_stage(inst, x_nom, eps0, Hn)
eval_rob_0 = evaluate_second_stage(inst, x_rob, eps0, Hn)
print(f"No disruption — Nominal profit: {eval_nom_0['profit']:,.1f}")
print(f"No disruption — Robust  profit: {eval_rob_0['profit']:,.1f}")
print(f"Pre-disruption cost of robustness: {eval_nom_0['profit']-eval_rob_0['profit']:,.1f}")

## 5. Worst-case disruption comparison

For each first-stage solution we find its **own worst-case disruption**
$\varepsilon^*$ by solving the adversary's problem (Corollary 2, eq. 17):

$$\varepsilon^*(x) = \arg\min_{\varepsilon \in \Xi(\Gamma)}\ Q(x, \varepsilon) \tag{17}$$

We then perform a **cross-evaluation**: each solution is also tested under the
other's worst-case disruption.  This reveals whether the nominal design is
catastrophically hurt by the disruption that the robust solution was designed
to withstand.

The **Value of Robustification (VOR)** is defined as:
$$\text{VOR} = \pi^{\rm rob}(\varepsilon^*_{\rm rob}) - \pi^{\rm nom}(\varepsilon^*_{\rm nom})$$

A positive VOR means the robust solution achieves higher guaranteed profit
under adversarial disruptions.

In [ ]:
eps_wc_nom, rc_wc_nom = worst_case_disruption(inst, x_nom, GAMMA, Hn)
eps_wc_rob, rc_wc_rob = worst_case_disruption(inst, x_rob, GAMMA, Hn)

# Each solution under its own worst-case
eval_nom_wc_nom = evaluate_second_stage(inst, x_nom, eps_wc_nom, Hn)
eval_rob_wc_rob = evaluate_second_stage(inst, x_rob, eps_wc_rob, Hn)

# Cross-evaluation
eval_nom_wc_rob = evaluate_second_stage(inst, x_nom, eps_wc_rob, Hn)
eval_rob_wc_nom = evaluate_second_stage(inst, x_rob, eps_wc_nom, Hn)

VOR = eval_rob_wc_rob['profit'] - eval_nom_wc_nom['profit']
print(f"Nominal profit under its own ε*: {eval_nom_wc_nom['profit']:,.1f}")
print(f"Robust  profit under its own ε*: {eval_rob_wc_rob['profit']:,.1f}")
print(f"Value of Robustification (VOR):  {VOR:,.1f}")
print()
print(f"Cross-eval — Nominal under rob's ε*: {eval_nom_wc_rob['profit']:,.1f}")
print(f"Cross-eval — Robust  under nom's ε*: {eval_rob_wc_nom['profit']:,.1f}")

## 6. Average-case comparison

Worst-case analysis gives a conservative bound.  To assess average performance
we draw $N$ random disruption scenarios $\varepsilon^{(k)} \in \Xi(\Gamma)$
and evaluate each solution under each scenario (Corollary 1, eq. 15).

The sampling procedure assigns random disruption levels $h_j \in H$ to
facilities sequentially, respecting the budget constraint
$\sum_j \sum_h \frac{h}{H-1}\,\varepsilon_{jh} \le \Gamma$ (paper eq. 11).

This Monte Carlo average approximates:
$$\bar{\pi} = \mathbb{E}_{\varepsilon \sim \Xi}[Q(x, \varepsilon)]$$

In [ ]:
scenarios = sample_disruptions(inst, GAMMA, Hn, N_SAMPLES, seed=SEED)

profits_nom_avg = []
profits_rob_avg = []
for eps in scenarios:
    r_nom = evaluate_second_stage(inst, x_nom, eps, Hn)
    r_rob = evaluate_second_stage(inst, x_rob, eps, Hn)
    profits_nom_avg.append(r_nom['profit'])
    profits_rob_avg.append(r_rob['profit'])

print(f"Average profit (N={N_SAMPLES}) — Nominal: {np.mean(profits_nom_avg):,.1f}  Robust: {np.mean(profits_rob_avg):,.1f}")

## 7. Summary table

In [ ]:
summary_data = {
    'Nominal profit': [
        eval_nom_0['profit'],
        eval_nom_wc_nom['profit'],
        np.mean(profits_nom_avg),
    ],
    'Robust profit': [
        eval_rob_0['profit'],
        eval_rob_wc_rob['profit'],
        np.mean(profits_rob_avg),
    ],
}
summary_df = pd.DataFrame(
    summary_data,
    index=['No disruption', 'Worst-case (own ε*)', f'Average case (N={N_SAMPLES})'],
)
summary_df['Difference (Robust-Nominal)'] = (
    summary_df['Robust profit'] - summary_df['Nominal profit']
)

print("=" * 70)
print("Profit comparison: Nominal vs Robust")
print("=" * 70)
print(summary_df.to_string(float_format=lambda x: f"{x:,.1f}"))
print()
print(f"Value of Robustification (VOR): {VOR:,.1f}")
print(f"Pre-disruption cost of robustness: {eval_nom_0['profit']-eval_rob_0['profit']:,.1f}")

# ---- Cost breakdown under worst-case ----
print()
print("=" * 70)
print("Cost breakdown under worst-case disruption")
print("=" * 70)
bd_nom = eval_nom_wc_nom['breakdown']
bd_rob = eval_rob_wc_rob['breakdown']
breakdown_df = pd.DataFrame(
    {
        'Nominal': [
            bd_nom['fixed_cost'],
            bd_nom['transport'],
            bd_nom['revenue'],
            bd_nom['congestion'],
            bd_nom['profit'],
        ],
        'Robust': [
            bd_rob['fixed_cost'],
            bd_rob['transport'],
            bd_rob['revenue'],
            bd_rob['congestion'],
            bd_rob['profit'],
        ],
    },
    index=['Fixed cost', 'Transport cost', 'Revenue', 'Congestion cost', 'Profit'],
)
print(breakdown_df.to_string(float_format=lambda x: f"{x:,.1f}"))

## 8. Sensitivity: VOR vs uncertainty budget $\Gamma$

We repeat the analysis for $\Gamma \in \{0, 1, 2, 3, 4\}$ to show how the
**Value of Robustification** and the **pre-disruption cost of robustness**
vary with the uncertainty budget.

A larger $\Gamma$ means the adversary can disrupt more facilities, so:
- The robust solution invests more in resilience (higher fixed cost), increasing
  the pre-disruption cost.
- The VOR tends to grow, since the nominal solution deteriorates faster under
  more severe disruptions.

At $\Gamma=0$ there are no disruptions and VOR = 0 (both solutions coincide or
are equally evaluated under no disruption).

Note: the C&CG solver is re-run for each $\Gamma$; the nominal solution is
reused as warm-start throughout.

In [ ]:
sensitivity_rows = []

for gamma in GAMMA_RANGE:
    print(f"--- Gamma={gamma} ---")

    # Solve robust problem for this Gamma
    ccg_g = solve_CCG(
        inst, gamma, Hn,
        x_init=x_nom,
        tol=TOL,
        time_limit=TIME_LIMIT,
        L_init=-abs(nom['profit'])*2,
        verbose=False,
    )
    x_rob_g = ccg_g['x_jr']
    print(f"  Robust LB: {ccg_g['profit_LB']:,.1f}  converged={ccg_g['converged']}  ({ccg_g['runtime']:.1f}s)")

    # No-disruption profits
    eps0_g = no_disruption_scenario(inst, Hn)
    e_nom0 = evaluate_second_stage(inst, x_nom,   eps0_g, Hn)
    e_rob0 = evaluate_second_stage(inst, x_rob_g, eps0_g, Hn)

    # Worst-case disruptions (each solution's own worst case)
    if gamma == 0:
        # No disruption possible; worst case == no disruption
        e_nom_wc = e_nom0
        e_rob_wc = e_rob0
    else:
        eps_wc_g_nom, _ = worst_case_disruption(inst, x_nom,   gamma, Hn)
        eps_wc_g_rob, _ = worst_case_disruption(inst, x_rob_g, gamma, Hn)
        e_nom_wc = evaluate_second_stage(inst, x_nom,   eps_wc_g_nom, Hn)
        e_rob_wc = evaluate_second_stage(inst, x_rob_g, eps_wc_g_rob, Hn)

    vor_g   = e_rob_wc['profit'] - e_nom_wc['profit']
    cost_g  = e_nom0['profit']   - e_rob0['profit']
    print(f"  VOR={vor_g:,.1f}  pre-disruption cost={cost_g:,.1f}")

    sensitivity_rows.append({
        'Gamma':                    gamma,
        'Nominal WC profit':        e_nom_wc['profit'],
        'Robust WC profit':         e_rob_wc['profit'],
        'VOR':                      vor_g,
        'Nominal ND profit':        e_nom0['profit'],
        'Robust ND profit':         e_rob0['profit'],
        'Pre-disruption cost':      cost_g,
    })

sens_df = pd.DataFrame(sensitivity_rows).set_index('Gamma')
print()
print(sens_df.to_string(float_format=lambda x: f"{x:,.1f}"))

## 9. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ---- Plot 1: profit comparison across scenarios ----
ax1 = axes[0]
scenarios_labels = ['No disruption', 'Worst-case', f'Average (N={N_SAMPLES})']
nom_profits = [
    eval_nom_0['profit'],
    eval_nom_wc_nom['profit'],
    np.mean(profits_nom_avg),
]
rob_profits = [
    eval_rob_0['profit'],
    eval_rob_wc_rob['profit'],
    np.mean(profits_rob_avg),
]

x_pos  = np.arange(len(scenarios_labels))
width  = 0.35
bars_n = ax1.bar(x_pos - width/2, nom_profits, width, label='Nominal',
                 color='steelblue', alpha=0.85)
bars_r = ax1.bar(x_pos + width/2, rob_profits, width, label='Robust',
                 color='darkorange', alpha=0.85)

ax1.set_xticks(x_pos)
ax1.set_xticklabels(scenarios_labels, fontsize=10)
ax1.set_ylabel('Profit')
ax1.set_xlabel('Scenario')
ax1.legend(fontsize=10)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')

# ---- Plot 2: VOR and pre-disruption cost vs Gamma ----
ax2 = axes[1]
gammas = sens_df.index.tolist()
ax2.plot(gammas, sens_df['VOR'],                color='darkorange', marker='o',
         linewidth=2, label='VOR (worst-case)')
ax2.plot(gammas, sens_df['Pre-disruption cost'], color='steelblue',  marker='s',
         linewidth=2, linestyle='--', label='Pre-disruption cost of robustness')
ax2.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Uncertainty budget $\\Gamma$')
ax2.set_ylabel('Profit difference')
ax2.set_xticks(gammas)
ax2.legend(fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

plt.tight_layout()
plt.savefig('fig_robustification.pdf', bbox_inches='tight')
plt.savefig('fig_robustification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figures saved: fig_robustification.pdf / .png")

## 10. Interpretation

**Pre-disruption cost of robustness.**  The nominal solution is tuned to
maximise profit when no disruptions occur.  Because the robust solution
hedges by opening facilities with higher capacity (or at different locations)
it typically achieves slightly *lower* profit under $\varepsilon_0$.  This
"insurance premium" is the pre-disruption cost of robustness.

**Value of Robustification (VOR).**  Under the adversarial worst-case
disruption, the nominal design can suffer severe profit losses because it was
never designed to handle capacity reductions.  The robust design, by
construction (eq. 12), minimises this downside.  A positive VOR confirms that
the robust solution provides a meaningfully higher guaranteed profit floor.

**Sensitivity to $\Gamma$.**  As the uncertainty budget grows:
- The adversary can inflict heavier damage on the nominal solution, so the
  **nominal worst-case profit falls steeply**.
- The robust solution invests more in resilience, **increasing the
  pre-disruption cost** while maintaining a higher profit floor.
- The **VOR therefore increases with $\Gamma$**: the benefit of robustification
  is largest when disruptions can be severe.

**Average-case results.**  The Monte Carlo evaluation (Section 3.2) shows
that under typical (non-adversarial) disruptions the robust solution often
performs comparably to or better than the nominal solution — the pre-disruption
cost of robustness is small relative to the VOR.

Together these results justify the use of the robust formulation eq. (12)
whenever disruptions are possible: a modest sacrifice in best-case profit
yields a substantially higher guaranteed profit under adversarial conditions.